# Electricity Demand Forecasting — Panama 2015–2020

This notebook covers:
1. **Dataset Overview** — shape, date range, missing values, target distribution
2. **Temporal Patterns** — hour, weekday, month, and heatmap
3. **Feature Correlations** — weather vs demand, holiday/school effects
4. **Stationarity Analysis** — ACF/PACF, ADF test, seasonal decomposition
5. **Model Results** — load saved metrics and display comparison plots

In [ ]:
import sys
from pathlib import Path

# Make src/ importable from the notebooks/ directory
ROOT_DIR = Path().resolve().parent
sys.path.insert(0, str(ROOT_DIR))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.features import load_data, engineer_features, temporal_train_test_split, TARGET_COL
from src.utils import OUTPUTS_DIR, load_json

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('Setup complete.')

---
## Section 1: Dataset Overview

In [ ]:
df_raw = load_data()
print(f'Shape: {df_raw.shape}')
print(f'Date range: {df_raw.index.min()} → {df_raw.index.max()}')
print(f'\nMissing values per column:')
print(df_raw.isnull().sum())
df_raw.head()

In [ ]:
print('Target variable (nat_demand) statistics:')
print(df_raw[TARGET_COL].describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histogram
axes[0].hist(df_raw[TARGET_COL], bins=80, color='steelblue', edgecolor='white', linewidth=0.3)
axes[0].set_xlabel('Demand (MW)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of National Electricity Demand')

# Full time series
axes[1].plot(df_raw.index, df_raw[TARGET_COL], linewidth=0.4, color='steelblue', alpha=0.8)
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Demand (MW)')
axes[1].set_title('Full Demand History (2015–2020)')

plt.tight_layout()
plt.show()

---
## Section 2: Temporal Patterns

In [ ]:
df = engineer_features(df_raw)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Hour of day
hourly = df.groupby('hour')[TARGET_COL].mean()
axes[0].plot(hourly.index, hourly.values, marker='o', color='steelblue', markersize=4)
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Avg Demand (MW)')
axes[0].set_title('Average Demand by Hour')
axes[0].set_xticks(range(0, 24, 3))

# Day of week
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
daily = df.groupby('day_of_week')[TARGET_COL].mean()
axes[1].bar(days, daily.values, color='steelblue', edgecolor='white')
axes[1].set_xlabel('Day of Week')
axes[1].set_ylabel('Avg Demand (MW)')
axes[1].set_title('Average Demand by Day of Week')

# Month
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly = df.groupby('month')[TARGET_COL].mean()
axes[2].bar(months, monthly.values, color='darkorange', edgecolor='white')
axes[2].set_xlabel('Month')
axes[2].set_ylabel('Avg Demand (MW)')
axes[2].set_title('Average Demand by Month')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: hour vs day_of_week
pivot = df.groupby(['hour', 'day_of_week'])[TARGET_COL].mean().unstack()
pivot.columns = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(
    pivot, ax=ax, cmap='YlOrRd', annot=False,
    cbar_kws={'label': 'Avg Demand (MW)'}, linewidths=0.3
)
ax.set_xlabel('Day of Week')
ax.set_ylabel('Hour of Day')
ax.set_title('Average Demand: Hour of Day vs Day of Week')
plt.tight_layout()
plt.show()

---
## Section 3: Feature Correlations

In [ ]:
from src.features import get_feature_columns

feature_cols = get_feature_columns(df)
corr_cols = feature_cols + [TARGET_COL]
corr = df[corr_cols].corr()[[TARGET_COL]].drop(TARGET_COL).sort_values(TARGET_COL, ascending=False)

fig, ax = plt.subplots(figsize=(7, 10))
sns.heatmap(
    corr, ax=ax, annot=True, fmt='.2f', cmap='RdBu_r',
    vmin=-1, vmax=1, center=0, cbar_kws={'shrink': 0.5}
)
ax.set_title(f'Feature Correlations with {TARGET_COL}')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter: temperature vs demand (all 3 cities)
temp_cols = ['T2M_toc', 'T2M_san', 'T2M_dav']
city_names = ['Tocumen', 'Santiago', 'David']
colors = ['steelblue', 'darkorange', 'seagreen']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, city, color in zip(axes, temp_cols, city_names, colors):
    sample = df.sample(n=min(5000, len(df)), random_state=42)
    ax.scatter(sample[col], sample[TARGET_COL], alpha=0.2, s=4, color=color)
    ax.set_xlabel(f'Temperature (°C) — {city}')
    ax.set_ylabel('Demand (MW)')
    ax.set_title(f'{city}: Temp vs Demand')
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots: holiday / school day effects
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

df['Holiday'] = df['holiday'].map({0: 'Non-Holiday', 1: 'Holiday'})
df['SchoolDay'] = df['school'].map({0: 'No School', 1: 'School Day'})

df.boxplot(column=TARGET_COL, by='Holiday', ax=axes[0], patch_artist=True)
axes[0].set_title('Demand: Holiday vs Non-Holiday')
axes[0].set_xlabel('')
axes[0].set_ylabel('Demand (MW)')

df.boxplot(column=TARGET_COL, by='SchoolDay', ax=axes[1], patch_artist=True)
axes[1].set_title('Demand: School Day vs No School')
axes[1].set_xlabel('')
axes[1].set_ylabel('Demand (MW)')

plt.suptitle('')
plt.tight_layout()
plt.show()

df.drop(columns=['Holiday', 'SchoolDay'], inplace=True)

---
## Section 4: Stationarity Analysis

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose

# Use a sample for speed (first 2 weeks)
sample_series = df[TARGET_COL].iloc[:24*14]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(sample_series, lags=72, ax=axes[0], alpha=0.05)
axes[0].set_title('ACF — nat_demand (first 2 weeks, up to lag 72h)')
plot_pacf(sample_series, lags=72, ax=axes[1], alpha=0.05, method='ywm')
axes[1].set_title('PACF — nat_demand')
plt.tight_layout()
plt.show()

In [ ]:
# Augmented Dickey-Fuller test
adf_result = adfuller(df[TARGET_COL].dropna(), autolag='AIC')
print('Augmented Dickey-Fuller Test')
print(f'  ADF Statistic : {adf_result[0]:.4f}')
print(f'  p-value       : {adf_result[1]:.6f}')
print(f'  Critical values:')
for k, v in adf_result[4].items():
    print(f'    {k}: {v:.4f}')
if adf_result[1] < 0.05:
    print('\n=> Series is STATIONARY (reject unit root hypothesis)')
else:
    print('\n=> Series is NON-STATIONARY (fail to reject unit root)')

In [ ]:
# Seasonal decomposition (daily averages for tractability)
daily = df[TARGET_COL].resample('D').mean().dropna()
decomp = seasonal_decompose(daily, model='additive', period=7)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
components = [
    (decomp.observed, 'Observed', 'black'),
    (decomp.trend, 'Trend', 'steelblue'),
    (decomp.seasonal, 'Seasonal (weekly)', 'darkorange'),
    (decomp.resid, 'Residual', 'gray'),
]
for ax, (data, title, color) in zip(axes, components):
    ax.plot(data.index, data.values, color=color, linewidth=0.8)
    ax.set_ylabel(title)
    ax.set_title(title)
axes[-1].set_xlabel('Date')
plt.suptitle('Seasonal Decomposition of Daily Demand (additive, period=7)', y=1.01)
plt.tight_layout()
plt.show()

---
## Section 5: Model Results

> Run `python -m src.train_lstm` and `python -m src.train_sarima` before this section.

In [ ]:
import json

lstm_metrics_path = OUTPUTS_DIR / 'lstm_metrics.json'
sarima_metrics_path = OUTPUTS_DIR / 'sarima_metrics.json'
comparison_path = OUTPUTS_DIR / 'model_comparison.json'

if lstm_metrics_path.exists() and sarima_metrics_path.exists():
    lstm_m = load_json(lstm_metrics_path)
    sarima_m = load_json(sarima_metrics_path)

    print('LSTM Metrics:')
    for k, v in lstm_m.items():
        print(f'  {k}: {v}')

    print('\nSARIMA Metrics:')
    for k, v in sarima_m.items():
        print(f'  {k}: {v}')

    # Comparison table as DataFrame
    rows = []
    for key in ['MAE_MW', 'RMSE_MW', 'MAPE_pct', 'R2', 'within_5pct', 'within_10pct']:
        rows.append({'Metric': key, 'LSTM': lstm_m.get(key, '-'), 'SARIMA': sarima_m.get(key, '-')})
    import pandas as pd
    pd.DataFrame(rows).set_index('Metric')
else:
    print('Metrics not found. Run training scripts first.')

In [ ]:
# Display saved plots
from IPython.display import Image, display

plot_files = [
    OUTPUTS_DIR / 'fig1_full_test_period.png',
    OUTPUTS_DIR / 'fig2_two_week_zoom.png',
    OUTPUTS_DIR / 'fig3_scatter.png',
    OUTPUTS_DIR / 'fig4_residuals.png',
    OUTPUTS_DIR / 'fig5_error_distribution.png',
]

for path in plot_files:
    if path.exists():
        print(f'\n{path.name}')
        display(Image(filename=str(path), width=900))
    else:
        print(f'{path.name} — not found, run evaluate.py first')

---
## Key Findings

1. **Strong daily and weekly seasonality**: Demand peaks consistently during morning (7–9am) and evening (6–9pm) hours, and is lower on weekends — captured by the lag-24 and lag-168 features.

2. **Temperature has a U-shaped relationship with demand**: Both very high and very low temperatures increase demand (cooling and heating loads), making temperature one of the strongest predictors alongside the lag features.

3. **Holiday and school effects are significant**: Demand on holidays drops noticeably relative to equivalent weekdays, confirming that human activity scheduling drives a material share of electricity consumption.

4. **LSTM outperforms SARIMA on short-term hourly patterns**: The SARIMA baseline operates on daily aggregates, losing intra-day resolution. The LSTM, trained on 168-hour sequences with all weather features, captures finer-grained temporal dynamics.

5. **Temporal train/test split is essential**: Using random splits on a time series leaks future information — all metrics reported here use a strict cutoff (train: 2015–2019, test: 2020).